In [9]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random() < 0.05:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.01, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'hour': random.randint(0, 5),
            'timestamp': datetime.now().isoformat(),
        }
    else:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'hour': random.randint(6, 23),
            'timestamp': datetime.now().isoformat(),
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value

    if tx['amount'] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Writing consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value

    # dodanie poziomu ryzyka
    if tx['amount'] > 3000:
        tx['risk_level'] = 'HIGH'
    elif tx['amount'] > 1000:
        tx['risk_level'] = 'MEDIUM'
    else:
        tx['risk_level'] = 'LOW'

    print(tx)

Writing consumer_enrich.py


In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    tx = message.value

    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1

    total_amount[store] += amount

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n=== PODSUMOWANIE ===")
        print(f"Przetworzono: {msg_count} wiadomości")
        print(f"{'Sklep':<12} {'Liczba':<10} {'Suma (PLN)':<15}")
        print("-" * 40)

        for sklep in store_counts:
            print(f"{sklep:<12} {store_counts[sklep]:<10} {total_amount[sklep]:<15.2f}")

Writing consumer_count.py


In [5]:
from datetime import datetime

def score_transaction(tx):
    score = 0
    rules = []

    # R1: amount > 3000
    if tx.get('amount', 0) > 3000:
        score += 3
        rules.append('R1')

    # R2: elektronika i amount > 1500
    if tx.get('category') == 'elektronika' and tx.get('amount', 0) > 1500:
        score += 2
        rules.append('R2')

    # R3: godzina < 6
    if 'hour' in tx:
        hour = tx['hour']
    else:
        hour = datetime.fromisoformat(tx['timestamp']).hour

    if hour < 6:
        score += 2
        rules.append('R3')

    return score, rules

In [6]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def score_transaction(tx):
    score = 0
    rules = []

    if tx.get('amount', 0) > 3000:
        score += 3
        rules.append('R1')

    if tx.get('category') == 'elektronika' and tx.get('amount', 0) > 1500:
        score += 2
        rules.append('R2')

    if 'hour' in tx:
        hour = tx['hour']
    else:
        hour = datetime.fromisoformat(tx['timestamp']).hour

    if hour < 6:
        score += 2
        rules.append('R3')

    return score, rules

for message in consumer:
    tx = message.value
    score, rules = score_transaction(tx)

    if score >= 3:
        alert = {
            'tx_id': tx['tx_id'],
            'user_id': tx['user_id'],
            'amount': tx['amount'],
            'store': tx['store'],
            'category': tx['category'],
            'timestamp': tx['timestamp'],
            'score': score,
            'rules': rules,
            'status': 'PODEJRZANA'
        }

        if 'hour' in tx:
            alert['hour'] = tx['hour']

        alert_producer.send('alerts', value=alert)
        alert_producer.flush()

        print(f"ALERT: {alert['tx_id']} | score={score} | rules={rules} | amount={alert['amount']:.2f} PLN")

Writing scoring_consumer.py


In [7]:
%%file consumer_alerts.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'alerts',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='alerts-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    print(message.value)

Writing consumer_alerts.py
